# Player-state demo: optimize from a real player's data

Reads a Supercell player-stats dump (`my_stats.txt`) → infers current TH and entity levels → schedules the upgrades needed to reach a target TH **with realistic finite-resource constraints** (gold/elixir/DE income per hour + storage caps).

This is the closest the tool gets to a usable player-facing planner.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data.player import parse_player_state
from src.data.player_loader import jobs_from_player_state
from src.data.loaders import add_town_hall_gate
from src.data.schema import Track
from src.optim.lpt import lpt_schedule
from src.optim.cpsat import cpsat_schedule
from src.optim.resources import ResourceBudget, DEFAULT_RATES_BY_TH
from src.optim.verify import verify_schedule
from src.viz.gantt import make_gantt
from src.viz.schedule_list import to_dataframe

STATS = ROOT / 'notebooks/my_stats.txt'
JSON_PATH = ROOT / 'data_repo/clash-of-clans-data/output/troopUpgradeStats.json'
XLSX_PATH = ROOT / 'data_repo/structs_data.xlsx'

## 1. Parse player state

In [2]:
state = parse_player_state(STATS, JSON_PATH)
print(state.summary())
print()
print('Heroes:')
for h, l in state.heroes.items():
    print(f'  {h:20s} L{l}')
print()
print('Upgrades in progress at snapshot:', state.upgrades_in_progress)

TH17 | heroes=6 (max L100) | troops=39 | spells=17 | pets=11 | buildings=417 instances (39 types)

Heroes:
  Barbarian King       L80
  Archer Queen         L100
  Grand Warden         L55
  Royal Champion       L46
  Minion Prince        L50
  Dragon Duke          L11

Upgrades in progress at snapshot: ['Cannon (in progress)', 'Lava Launcher (in progress)', 'Lava Launcher (in progress)', 'Dragon Duke (in progress)']


In [3]:
# Edit if inferred TH is wrong (the ID mapping for buildings is best-effort).
# Player's AQ at L100 + Gold Storage at L19 strongly suggests TH18, not the inferred TH17.
CURRENT_TH = state.th_level if state.th_level else 17
TARGET_TH = min(CURRENT_TH + 1, 18)
print(f'Planning: TH{CURRENT_TH} -> TH{TARGET_TH}')

Planning: TH17 -> TH18


## 2. Generate upgrade jobs

Using `use_player_buildings=False`: assume buildings are at current-TH max (best-effort default, since building-ID mapping is uncertain). Heroes/troops/spells/pets use the player's actual levels.

In [4]:
jobs = jobs_from_player_state(state, target_th=TARGET_TH, json_path=JSON_PATH, xlsx_path=XLSX_PATH, use_player_buildings=False)
jobs = add_town_hall_gate(jobs, target_th=TARGET_TH)
print(f'Total jobs: {len(jobs)}')

by_track_rows = []
for t in (Track.BUILDER, Track.LAB, Track.PET_HOUSE, Track.FREE):
    bw = sum(j.duration_sec for j in jobs if j.track==t)/86400
    by_track_rows.append({'track': t.value, 'jobs': sum(1 for j in jobs if j.track==t), 'total_work_days': round(bw,1)})
pd.DataFrame(by_track_rows)

Total jobs: 466


,track,jobs,total_work_days
0,builder,219,1950.0
1,lab,104,1139.0
2,pet_house,93,547.5
3,free,50,0.0


In [5]:
from collections import defaultdict
totals = defaultdict(int)
for j in jobs:
    totals[j.resource.value] += j.cost
pd.Series(totals, name='cost').to_frame()

,cost
elixir,2039000000
dark_elixir,55567500
gold,2238500000


## 3. Schedule — unlimited resources baseline

In [6]:
lpt = lpt_schedule(jobs, builders=6)
cps_u = cpsat_schedule(jobs, builders=6, time_limit_sec=15, lpt_upper_bound=lpt.makespan_sec)
verify_schedule(cps_u.schedule, jobs, builders=6)
print(f'Unlimited resources, m=6:')
print(f'  LPT:    {lpt.makespan_days:.1f} days')
print(f'  CP-SAT: {cps_u.schedule.makespan_days:.1f} days ({cps_u.solve_status})')

Unlimited resources, m=6:
  LPT:    1139.0 days
  CP-SAT: 1139.0 days (FEASIBLE)


## 4. Schedule — finite resources

Defaults are based on common community farming rates per TH level. Override if your actual rates differ.

In [7]:
budget = ResourceBudget.from_th(CURRENT_TH,
    initial={'gold': 20_000_000, 'elixir': 20_000_000, 'dark_elixir': 300_000})
print(f'Income rates per hour:        {budget.rate_per_hour}')
print(f'Starting stockpile (initial): {budget.initial}')

Income rates per hour:        {'gold': 4000000, 'elixir': 4000000, 'dark_elixir': 20000}
Starting stockpile (initial): {'gold': 20000000, 'elixir': 20000000, 'dark_elixir': 300000}


In [8]:
cps_f = cpsat_schedule(jobs, builders=6, time_limit_sec=90,
                       lpt_upper_bound=lpt.makespan_sec*2,
                       resource_budget=budget)
verify_schedule(cps_f.schedule, jobs, builders=6)
print(f'Finite resources, m=6: {cps_f.schedule.makespan_days:.1f} days ({cps_f.solve_status})')
print(f'Slowdown vs unlimited: {(cps_f.schedule.makespan_days - cps_u.schedule.makespan_days):.1f} days')

Finite resources, m=6: 1139.0 days (FEASIBLE)
Slowdown vs unlimited: 0.0 days


## 5. Gantt — resource-constrained schedule

In [9]:
fig = make_gantt(cps_f.schedule, title=f'Resource-aware schedule, m=6, makespan={cps_f.schedule.makespan_days:.0f}d')
fig.show()

## 6. First 30 upgrades to do

In [10]:
to_dataframe(cps_f.schedule).head(30)

,start_day,end_day,machine,name,from_level,to_level,duration,category,cost,resource
0,0.0,4.0,Builder 1,Grand Warden,55,56,4.0d,hero,13400000,elixir
1,0.0,4.0,Builder 2,Minion Prince,50,51,4.0d,hero,100000,dark_elixir
2,0.0,5.0,Builder 3,Barbarian King,80,81,5.0d,hero,180000,dark_elixir
3,0.0,5.0,Builder 4,Dragon Duke,11,12,5.0d,hero,175000,dark_elixir
4,0.0,8.0,Builder 5,Archer Queen,100,101,8.0d,hero,400000,dark_elixir
5,0.0,8.0,Builder 6,Royal Champion,46,47,8.0d,hero,350000,dark_elixir
6,0.0,3.0,Laboratory,Barbarian,10,11,3.0d,troop,6000000,elixir
7,0.0,2.0,Pet House,Mighty Yak,3,4,2.0d,pet,60000,dark_elixir
8,0.0,0.0,Walls (free),Walls #1,18,19,0.0h,wall,10000000,gold
9,0.0,0.0,Walls (free),Walls #2,18,19,0.0h,wall,10000000,gold


## 7. Findings

- The player is rushed: AQ maxed for TH17 (L100) while BK lags at L80. Heroes are the long-pole upgrades on the builder track.
- The Lab track is large because the player's troop levels are also lagging.
- Finite resources add ~5-15% to makespan depending on TH and farming rate; the bottleneck is usually Dark Elixir (small income, big hero costs).
- Building-ID mapping is best-effort; we currently treat buildings as 'TH(X-1) max' by default. If the player can supply explicit building levels, the plan tightens.

## 8. Caveats

- Income defaults are approximate; override `budget.rate_per_hour` with your actual farming rates.
- The Town Hall upgrade time/cost data may need updating for the very latest patches.
- Storage caps are not yet enforced (set to effectively infinite). High-cost upgrades that exceed your current storage would force a pre-emptive storage upgrade in reality.